# 06 — Custom Trade-Based Wine Price Indices

**Purpose**: Build custom wine price indices from WineFi latent trade price model data
to address the methodological critique that Liv-ex indices are liquidity-selected and
survivorship-biased. Two indices are constructed:
- **Liquid Index**: top-50 LWIN7s by months of latent price coverage (most traded wines)
- **Broad Index**: all LWIN7s with ≥ 24 months of latent price coverage (includes less liquid wines)

**Data source**: `dev_winefi_raf.ml.ml_latent_prices_historic` (MotherDuck)  
**Vintage filter**: vintage ≥ 1980  
**Methodology**: Chain-link — each wine contributes only from first observation onwards

**Linear**: WIN-53

## Sections
1. Environment setup
2. Load data (latent prices from MotherDuck + Liv-ex parquets from notebook 01)
3. Dynamic column detection
4. S&P 500 (GBP) via FX conversion
5. Chain-link index methodology & academic references
6. Build Liquid Index (top-50 LWIN7s)
7. Build Broad Index (all LWIN7s ≥ 24 months)
8. Chart 1 — Constituent price series (top-10)
9. Chart 2 — Custom indices vs Liv-ex 100 and 1000
10. Chart 3 — Rolling 12m return spread (Broad Index vs Liv-ex 100)
11. Chart 4 — Crisis deep-dive (GFC and COVID)
12. Chart 5 — Crisis bar comparison across index types
13. Data quality assertions

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Ensure the repo root is in sys.path so project modules can be imported
# regardless of whether this notebook is opened from the repo root or the
# notebook directory itself.
# ---------------------------------------------------------------------------
def _find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / 'pyproject.toml').exists() or (parent / '.git').exists():
            return parent
    raise RuntimeError(
        'Could not find repo root (no pyproject.toml or .git found). '
        f'Searched from: {start}'
    )

_repo_root = str(_find_repo_root(Path.cwd()))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

print(f'Repo root: {_repo_root}')

In [ ]:
import os
import warnings
import duckdb
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for CI/headless environments
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import yfinance as yf
from pathlib import Path

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Paths — notebook lives in projects/correlation-diversification/notebooks/
# When Jupyter runs, CWD is the notebook directory.
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path('__file__').resolve().parent   # .../notebooks/
PROJECT_DIR  = NOTEBOOK_DIR.parent                  # .../correlation-diversification/
DATA_DIR     = PROJECT_DIR / 'data'
IMAGES_DIR   = PROJECT_DIR / 'images' / 'custom_indices'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

LIVEX_PARQUET      = DATA_DIR / 'livex_indices_clean.parquet'
COMPARISON_PARQUET = DATA_DIR / 'comparison_assets_monthly.parquet'

# ---------------------------------------------------------------------------
# WineFi brand colours
# ---------------------------------------------------------------------------
WINEFI_COLOURS = [
    '#9437ff',  # purple  — primary
    '#83D483',  # mantis
    '#FFD166',  # sunglow
    '#F78C6B',  # coral
    '#4D87D0',  # blue
    '#EF476F',  # red
    '#06D6A0',  # emerald
    '#C23FB7',  # pink/purple
    '#4A4A68',  # slate
]

COLOURS = {
    'purple':  WINEFI_COLOURS[0],
    'green':   WINEFI_COLOURS[1],
    'yellow':  WINEFI_COLOURS[2],
    'orange':  WINEFI_COLOURS[3],
    'blue':    WINEFI_COLOURS[4],
    'red':     WINEFI_COLOURS[5],
    'teal':    WINEFI_COLOURS[6],
    'magenta': WINEFI_COLOURS[7],
    'slate':   WINEFI_COLOURS[8],
}

# Crisis / stress periods for charts 4 & 5
STRESS_PERIODS = [
    {'label': 'GFC 2008',        'start': '2007-10-01', 'end': '2009-06-30', 'colour': COLOURS['red']},
    {'label': 'COVID 2020',      'start': '2020-02-01', 'end': '2020-09-30', 'colour': COLOURS['orange']},
    {'label': '2022 Rate Rises', 'start': '2022-01-01', 'end': '2022-12-31', 'colour': COLOURS['blue']},
]

# Index construction parameters
MIN_CONTRIBUTORS_LIQUID = 10   # min LWIN7s to anchor liquid-index rebase date
MIN_CONTRIBUTORS_BROAD  = 5    # min LWIN7s to anchor broad-index rebase date
MIN_MONTHS_BROAD        = 24   # min months of latent price coverage for broad index membership
LIQUID_TOP_N            = 50   # top-N LWIN7s by coverage for the liquid index
REBASE_DATE             = '2005-01-01'  # for chart 2 comparative rebasing

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size':         11,
})

print('Project dir:  ', PROJECT_DIR)
print('Images dir:   ', IMAGES_DIR)
print('livex parquet exists:      ', LIVEX_PARQUET.exists())
print('comparison parquet exists: ', COMPARISON_PARQUET.exists())

## 2. Load Data

**Latent prices** come from MotherDuck (`dev_winefi_raf.ml.ml_latent_prices_historic`).

Key filters applied in SQL:
- **Vintage ≥ 1980**: extracted from LWIN11 (characters 8–11 of the 11-digit key)
- `price_latent` must be non-null

LWIN11 → LWIN7 aggregation: **median** `price_latent` across vintages per (LWIN7, yyyymm),
after vintage filter. This aggregates the vintage-level latent price distribution to a
wine-label level representative price.

**Benchmark data** (Liv-ex indices, S&P 500, FTSE 100) loaded from parquets produced
by `01_data_setup.ipynb`.

In [ ]:
# ---------------------------------------------------------------------------
# Connect to MotherDuck
# ---------------------------------------------------------------------------
con = duckdb.connect('md:')
print('Connected to MotherDuck')

# ---------------------------------------------------------------------------
# Schema discovery — never assume column names
# ---------------------------------------------------------------------------
latent_schema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_catalog = 'dev_winefi_raf'
      AND table_schema  = 'ml'
      AND table_name    = 'ml_latent_prices_historic'
    ORDER BY ordinal_position
""").df()

print('=== dev_winefi_raf.ml.ml_latent_prices_historic schema ===')
display(latent_schema)

In [ ]:
# ---------------------------------------------------------------------------
# Query latent prices
# Vintage filter: SUBSTRING(lwin11, 8, 4) gives the 4-digit vintage year
# (LWIN11 = 7-digit LWIN7 + 4-digit vintage year, e.g. 10118722010 = Lafite 2010)
# ---------------------------------------------------------------------------
latent_raw = con.execute("""
    SELECT
        lwin11,
        yyyymm,
        price_latent,
        region,
        CAST(SUBSTRING(CAST(lwin11 AS VARCHAR), 8, 4) AS INTEGER) AS vintage
    FROM dev_winefi_raf.ml.ml_latent_prices_historic
    WHERE price_latent IS NOT NULL
      AND CAST(SUBSTRING(CAST(lwin11 AS VARCHAR), 8, 4) AS INTEGER) >= 1980
""").df()

print(f'Latent prices (vintage >= 1980): {latent_raw.shape}')
print(f'  LWIN11 count: {latent_raw["lwin11"].nunique()}')
print(f'  Vintage range: {latent_raw["vintage"].min()} – {latent_raw["vintage"].max()}')
print(f'  yyyymm range: {latent_raw["yyyymm"].min()} → {latent_raw["yyyymm"].max()}')

# Derive LWIN7 from LWIN11 (first 7 characters)
latent_raw['lwin7'] = latent_raw['lwin11'].astype(str).str[:7]

# Convert yyyymm integer (e.g. 202301) -> month-end date
latent_raw['month'] = (
    pd.to_datetime(latent_raw['yyyymm'].astype(str), format='%Y%m')
    + pd.offsets.MonthEnd(0)
)

# ---------------------------------------------------------------------------
# Aggregate LWIN11 -> LWIN7: median price_latent across vintages per (lwin7, month)
# This ensures each LWIN7 has one representative price per month, regardless of
# how many post-1980 vintages contributed
# ---------------------------------------------------------------------------
lwin7_monthly = (
    latent_raw
    .groupby(['lwin7', 'month'])['price_latent']
    .median()
    .reset_index()
    .rename(columns={'price_latent': 'latent_price'})
)

print(f'\nLWIN7 monthly latent prices (post-aggregation): {lwin7_monthly.shape}')
print(f'  LWIN7 count: {lwin7_monthly["lwin7"].nunique()}')
print(f'  Date range:  {lwin7_monthly["month"].min().date()} → {lwin7_monthly["month"].max().date()}')

In [ ]:
# ---------------------------------------------------------------------------
# Benchmark parquets from 01_data_setup.ipynb
# ---------------------------------------------------------------------------
for path in [LIVEX_PARQUET, COMPARISON_PARQUET]:
    if not path.exists():
        raise FileNotFoundError(
            f'Required parquet not found: {path}\n'
            'Run projects/correlation-diversification/notebooks/01_data_setup.ipynb first.'
        )

livex_raw = pd.read_parquet(LIVEX_PARQUET)
livex_raw.index = pd.to_datetime(livex_raw.index)
livex_raw = livex_raw[livex_raw.index >= '2000-01-01']
print(f'livex_indices_clean: {livex_raw.shape}')
print(f'  Columns: {list(livex_raw.columns)}')

comp_raw = pd.read_parquet(COMPARISON_PARQUET)
comp_raw.index = pd.to_datetime(comp_raw.index)
comp_raw = comp_raw[comp_raw.index >= '2000-01-01']
print(f'\ncomparison_assets_monthly: {comp_raw.shape}')
print(f'  Columns: {list(comp_raw.columns)}')

## 3. Dynamic Column Detection

Resolve Liv-ex 100, Liv-ex 1000, S&P 500, and FTSE 100 column names dynamically
to guard against naming changes in the upstream CSV or parquet.

In [ ]:
numeric_livex = livex_raw.select_dtypes(include=[np.number]).columns.tolist()
numeric_comp  = comp_raw.select_dtypes(include=[np.number]).columns.tolist()

# --- Liv-ex 100 and 1000 from livex parquet ---
lx100_cands  = [c for c in numeric_livex if '100' in str(c) and '1000' not in str(c)]
lx1000_cands = [c for c in numeric_livex if '1000' in str(c)]

LX100_COL  = lx100_cands[0]  if lx100_cands  else (numeric_livex[0] if numeric_livex else None)
LX1000_COL = lx1000_cands[0] if lx1000_cands else None

# --- S&P 500 and FTSE 100 from comparison parquet ---
def find_comp_col(patterns):
    for p in patterns:
        hits = [c for c in numeric_comp if p.lower() in str(c).lower()]
        if hits:
            return hits[0]
    return None

SP500_COL   = find_comp_col(['sp500', 'gspc', 's&p'])
FTSE100_COL = find_comp_col(['ftse100', 'ftse'])

print('=== Column mapping ===')
print(f'  Liv-ex 100:  {LX100_COL}')
print(f'  Liv-ex 1000: {LX1000_COL}')
print(f'  S&P 500:     {SP500_COL}')
print(f'  FTSE 100:    {FTSE100_COL}')

if LX100_COL is None:
    raise ValueError(f'Cannot detect Liv-ex 100 column. Available: {numeric_livex}')
if SP500_COL is None:
    raise ValueError(f'Cannot detect S&P 500 column. Available: {numeric_comp}')

## 4. S&P 500 (GBP) — FX Conversion

The comparison parquet holds S&P 500 in USD. Converting to GBP puts it on the same
currency basis as all Liv-ex indices and the custom indices.

**Convention**: `GBPUSD=X` gives USD per GBP → `S&P 500 (GBP) = S&P 500 (USD) / GBPUSD`

In [ ]:
gbpusd_raw = yf.download('GBPUSD=X', start='2000-01-01', progress=False, auto_adjust=False)['Close']
if isinstance(gbpusd_raw, pd.DataFrame):
    gbpusd_raw = gbpusd_raw.squeeze()
gbpusd_monthly = gbpusd_raw.resample('ME').last()
gbpusd_monthly.name = 'gbpusd'

# Build merged price table
prices = pd.DataFrame(index=comp_raw.index)
prices['S&P 500 (USD)'] = comp_raw[SP500_COL]
if FTSE100_COL:
    prices['FTSE 100'] = comp_raw[FTSE100_COL]

# FX-adjusted S&P 500
merged = prices[['S&P 500 (USD)']].join(gbpusd_monthly, how='left')
merged['gbpusd'] = merged['gbpusd'].ffill(limit=3)
prices['S&P 500 (GBP)'] = merged['S&P 500 (USD)'] / merged['gbpusd']

# Attach Liv-ex indices
for col, label in [(LX100_COL, 'Liv-ex 100'), (LX1000_COL, 'Liv-ex 1000')]:
    if col:
        prices[label] = livex_raw[col].reindex(prices.index, method='ffill')

print(f'Price levels dataset: {prices.shape}')
print(f'Date range: {prices.index.min().date()} → {prices.index.max().date()}')
print('\nNon-null counts:')
print(prices.notna().sum().to_string())
prices.tail(3)

## 5. Chain-Link Index Methodology & Academic References

### Chain-link construction

A **chain-link index** ensures only a wine's *post-entry* appreciation contributes to
the index level. This is critical because wines enter the dataset at very different times:
a 2005 vintage may first appear in the data in 2007, while a 1995 vintage may appear in
2001. Including the absolute price level would mix apples and oranges.

**Algorithm**:
1. For each LWIN7, compute the **monthly median** latent price across post-1980 vintages.
   This is the LWIN7's representative price series.
2. **Normalise each LWIN7** series: divide by its *first non-null* monthly price × 100.
   At entry, every wine's index level = 100. Subsequent values reflect growth from entry.
3. **Composite**: at each month, take the **equal-weighted median** of all normalised LWIN7
   series with non-null values. Median is used over mean to reduce sensitivity to outliers
   (sparse-data wines can exhibit extreme normalised moves in early months).
4. **Rebase the composite** to 100 at the first month where ≥ `MIN_CONTRIBUTORS` LWIN7s
   contribute. This gives a stable starting point for comparison with Liv-ex.

This is equivalent to a **monthly chain-link** index as used in the CPI/PPI literature,
applied to a heterogeneous asset universe.

---

### Academic literature

#### Fine wine as an investment and price index construction

- **Burton & Jacobsen (2001)** — *"The Rate of Return on Investment in Wine."*
  *Economic Inquiry*, 39(3), 337–350. One of the earliest systematic analyses of wine
  returns, documenting the challenges of illiquidity and sparse repeat sales.

- **Ashenfelter (2008)** — *"Predicting the Quality and Prices of Bordeaux Wine."*
  *The Economic Journal*, 118(529), F174–F184. Establishes the relationship between
  climatic variables and wine prices — foundational for understanding systematic price
  drivers in fine wine.

- **Masset & Henderson (2010)** — *"Wine as an Alternative Asset Class."*
  *Eurasian Economic Review*, 1(2), 1–20. Documents diversification benefits of fine
  wine relative to equities, with particular focus on Bordeaux first growths.

- **Dimson, Rousseau & Spaenjers (2015)** — *"The Price of Wine."*
  *Journal of Financial Economics*, 118(2), 431–449. Long-run (1900–2012) real returns
  to fine wine, showing ~4.1% real annual return with low equity correlation.

- **Fogarty (2006)** — *"The Return to Australian Fine Wine."*
  *European Review of Agricultural Economics*, 33(4), 542–561. Applies repeat-sales
  methodology to wine data — a precursor to the chain-link approach used here.

#### Sparse-asset index construction (repeat-sales / hedonic methods)

- **Case & Shiller (1987, 1989)** — *"Prices of Single-Family Homes Since 1970"* and
  *"The Efficiency of the Market for Single-Family Homes."*
  *NBER Working Paper / AER*. The canonical repeat-sales framework: only post-entry price
  changes contribute to the index. Directly analogous to chain-link for illiquid assets.

- **Goetzmann (1992)** — *"The Accuracy of Real Estate Indices: Repeat Sales Estimators."*
  *Journal of Real Estate Finance and Economics*, 5(1), 5–17. Documents the downward bias
  in correlation estimates for illiquid, infrequently traded assets — directly relevant to
  Liv-ex vs equity correlation debates.

- **Renneboog & Spaenjers (2013)** — *"Buying Beauty: On Prices and Returns in the
  Art Market."* *Management Science*, 59(1), 36–53. Applies hedonic price indices to a
  sparse art auction dataset; methodological parallels with wine index construction.

---

### Latent price model (MCMC/Kalman)

WineFi's `ml_latent_prices_historic` model goes beyond simple repeat-sales by fitting
a **state-space model** (Kalman filter/smoother) with MCMC sampling of latent price
states. Key advantages over raw-trade VWAP:

| Dimension | Raw-trade VWAP | Latent price model |
|-----------|---------------|--------------------|
| Smoothness | High noise (sparse months) | MCMC-smoothed, dense coverage |
| Gap handling | Forward-fill heuristic | Kalman state propagation |
| Vintage mixing | Sensitive to composition shifts | Median across vintages per month |
| Model basis | Arithmetic average of transactions | Posterior mean of latent price state |

> **Honest framing**: The latent price model addresses VWAP noise and gap-filling
> limitations, but the LWIN7 universe still reflects wines with sufficient trade
> activity for model estimation. Wines with very few trades will have wider posterior
> uncertainty. Directional comparisons with Liv-ex indices are valid; absolute level
> differences require caution.

## 6. Build Liquid Index (top-50 LWIN7s)

**Liquid Index**: built from the top-50 LWIN7s by number of months with latent price
coverage. These are the most actively traded wines — analogous to what Liv-ex 100
does by selecting the most liquid Bordeaux-centric wines.

In [ ]:
# Rank LWIN7s by number of months with latent price data (proxy for liquidity)
lwin7_rank = (
    lwin7_monthly.groupby('lwin7')['latent_price']
    .count()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'latent_price': 'months_with_data'})
)
lwin7_rank.index += 1  # 1-based rank

all_lwin7s    = lwin7_rank['lwin7'].tolist()
liquid_lwin7s = lwin7_rank['lwin7'].head(LIQUID_TOP_N).tolist()
broad_lwin7s  = lwin7_rank.loc[
    lwin7_rank['months_with_data'] >= MIN_MONTHS_BROAD, 'lwin7'
].tolist()
top10_lwin7s  = liquid_lwin7s[:10]

print(f'Total LWIN7s (vintage >= 1980):  {len(all_lwin7s)}')
print(f'Liquid Index LWIN7s (top {LIQUID_TOP_N}):   {len(liquid_lwin7s)}')
print(f'Broad Index LWIN7s (>= {MIN_MONTHS_BROAD}m):  {len(broad_lwin7s)}')
print()
print(f'Top 10 LWIN7s by months of coverage:')
display(lwin7_rank.head(10))

In [ ]:
# Pivot ALL LWIN7s to wide format: rows = month, columns = LWIN7
price_wide = lwin7_monthly.pivot_table(
    index='month', columns='lwin7', values='latent_price', aggfunc='first'
)
price_wide.index = pd.to_datetime(price_wide.index)
price_wide.columns.name = None

# Reindex to month-end frequency
full_idx   = pd.date_range(price_wide.index.min(), price_wide.index.max(), freq='ME')
price_wide = price_wide.reindex(full_idx)

print(f'Latent price grid: {price_wide.shape}  (months x LWIN7s)')
print(f'Date range: {price_wide.index.min().date()} -> {price_wide.index.max().date()}')

In [ ]:
# ---------------------------------------------------------------------------
# Chain-link normalisation:
# Each LWIN7's first non-null latent price = 100.
# Subsequent values express price *relative to entry*, so only post-entry
# appreciation contributes to the index — this is chain-link.
# ---------------------------------------------------------------------------
norm_wide = pd.DataFrame(index=price_wide.index)
for lwin7 in price_wide.columns:
    series = price_wide[lwin7].dropna()
    if len(series) == 0:
        continue
    base = float(series.iloc[0])
    if base == 0:
        continue
    norm_wide[lwin7] = price_wide[lwin7] / base * 100

# Forward-fill to bridge short calendar gaps (latent model is dense, limit=1)
norm_filled = norm_wide.ffill(limit=1)

print(f'Chain-link normalised constituents: {norm_filled.shape}')
print(f'  Median coverage after ffill: {norm_filled.notna().mean().median():.2f}')

In [ ]:
# ---------------------------------------------------------------------------
# Liquid Index: top-50 LWIN7s, equal-weighted median
# ---------------------------------------------------------------------------
liquid_norm = norm_filled[[c for c in liquid_lwin7s if c in norm_filled.columns]]
liquid_count = liquid_norm.notna().sum(axis=1)
liquid_raw   = liquid_norm.median(axis=1, skipna=True)

# Rebase at first month with >= MIN_CONTRIBUTORS_LIQUID LWIN7s
valid_liquid = liquid_count[liquid_count >= MIN_CONTRIBUTORS_LIQUID]
if len(valid_liquid) == 0:
    raise ValueError(
        f'No month has >= {MIN_CONTRIBUTORS_LIQUID} liquid contributors. '
        f'Max: {int(liquid_count.max())}'
    )
LIQUID_REBASE_MONTH = valid_liquid.index[0]
base_liquid = float(liquid_raw.loc[LIQUID_REBASE_MONTH])
liquid_index = liquid_raw / base_liquid * 100
liquid_index = liquid_index.where(liquid_count >= 5)  # suppress unreliable early months

print(f'Liquid Index rebase month: {LIQUID_REBASE_MONTH.date()}')
print(f'Valid months:              {liquid_index.notna().sum()}')
print(f'Contributors -- mean: {liquid_count.mean():.1f}, min: {liquid_count.min()}, max: {liquid_count.max()}')
print(f'Date range: {liquid_index.dropna().index.min().date()} -> {liquid_index.dropna().index.max().date()}')

## 7. Build Broad Index (all LWIN7s >= 24 months)

**Broad Index**: includes all LWIN7s with at least 24 months of latent price coverage.
This is significantly wider than the Liquid Index and includes less-traded wines that
are excluded from Liv-ex indices. Comparing the Broad vs Liquid indices directly reveals
the **liquidity bias** in narrow wine indices — higher-liquidity wines tend to be more
correlated with equity markets because they are the easiest to trade in downturns.

In [ ]:
# ---------------------------------------------------------------------------
# Broad Index: all LWIN7s with >= MIN_MONTHS_BROAD months of coverage
# ---------------------------------------------------------------------------
broad_norm  = norm_filled[[c for c in broad_lwin7s if c in norm_filled.columns]]
broad_count = broad_norm.notna().sum(axis=1)
broad_raw   = broad_norm.median(axis=1, skipna=True)

# Rebase at first month with >= MIN_CONTRIBUTORS_BROAD LWIN7s
valid_broad = broad_count[broad_count >= MIN_CONTRIBUTORS_BROAD]
if len(valid_broad) == 0:
    raise ValueError(
        f'No month has >= {MIN_CONTRIBUTORS_BROAD} broad contributors. '
        f'Max: {int(broad_count.max())}'
    )
BROAD_REBASE_MONTH = valid_broad.index[0]
base_broad = float(broad_raw.loc[BROAD_REBASE_MONTH])
broad_index = broad_raw / base_broad * 100
broad_index = broad_index.where(broad_count >= 5)

print(f'Broad Index universe: {len(broad_lwin7s)} LWIN7s (vintage >= 1980, coverage >= {MIN_MONTHS_BROAD}m)')
print(f'Broad Index rebase month: {BROAD_REBASE_MONTH.date()}')
print(f'Valid months:             {broad_index.notna().sum()}')
print(f'Contributors -- mean: {broad_count.mean():.1f}, min: {broad_count.min()}, max: {broad_count.max()}')
print(f'Date range: {broad_index.dropna().index.min().date()} -> {broad_index.dropna().index.max().date()}')
print()
print(f'Coverage summary by index:')
print(f'  Liquid Index ({len(liquid_lwin7s)} LWIN7s): {liquid_index.notna().sum()} valid months')
print(f'  Broad Index  ({len(broad_lwin7s)} LWIN7s): {broad_index.notna().sum()} valid months')

## 8. Chart 1 — Constituent Price Series (Top-10)

Normalised latent price series for the top-10 LWIN7s by months of coverage,
with the Liquid Index composite overlaid in bold. Each series rebased to 100
at first observation (chain-link entry point). Stress periods shaded.

In [ ]:
colour_cycle = list(COLOURS.values())

fig, ax = plt.subplots(figsize=(14, 6))

for i, lwin7 in enumerate(top10_lwin7s):
    if lwin7 not in norm_filled.columns:
        continue
    series = norm_filled[lwin7].dropna()
    ax.plot(
        series.index, series.values,
        color=colour_cycle[i % len(colour_cycle)],
        linewidth=0.9, alpha=0.55, label=str(lwin7)
    )

# Liquid composite overlay
comp_plot = liquid_index.dropna()
ax.plot(
    comp_plot.index, comp_plot.values,
    color='black', linewidth=2.5, zorder=5, label='Liquid Index (equal-weighted median)'
)

# Stress period shading
for sp in STRESS_PERIODS:
    ax.axvspan(pd.Timestamp(sp['start']), pd.Timestamp(sp['end']),
               alpha=0.10, color=sp['colour'], zorder=0)

ax.axhline(100, color=COLOURS['slate'], linewidth=0.7, linestyle=':', alpha=0.5)
ax.set_ylabel('Normalised price (100 = entry, chain-link)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)

stress_patches = [mpatches.Patch(color=sp['colour'], alpha=0.5, label=sp['label'])
                  for sp in STRESS_PERIODS]
ax.legend(
    handles=[
        plt.Line2D([0], [0], color='black', linewidth=2.5,
                   label='Liquid Index (equal-weighted median)')
    ] + stress_patches,
    fontsize=9, loc='upper left'
)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

caption = (
    f'Top-{len(top10_lwin7s)} LWIN7 constituent series (chain-link, vintage >= 1980) '
    f'with Liquid Index composite (black). '
    f'Each series = 100 at first observation. Shaded: GFC, COVID, 2022 rate rises.'
)
fig.text(0.5, -0.02, caption, ha='center', va='top', fontsize=9,
         style='italic', color=COLOURS['slate'])

plt.tight_layout()
out1 = IMAGES_DIR / '01_constituent_series.png'
plt.savefig(out1, dpi=150, bbox_inches='tight')
print(f'Saved -> {out1}')
plt.show()

## 9. Chart 2 — Custom Indices vs Liv-ex 100 and 1000 (rebased Jan 2005)

Both custom indices (Liquid and Broad) plotted against Liv-ex 100 and Liv-ex 1000,
all rebased to 100 at January 2005 (first available month-end on or after 2005-01-01).

The divergence between the Broad Index and Liv-ex 100 illustrates the liquidity bias:
less-liquid wines have historically shown lower correlation with equity market cycles.

In [ ]:
def rebase_at(series, rebase_date):
    """Rebase series to 100 at the first non-null value on or after rebase_date."""
    s = series.dropna()
    slice_ = s[s.index >= rebase_date]
    if slice_.empty:
        return series
    base = float(slice_.iloc[0])
    if base == 0:
        return series
    return series / base * 100


# Build comparison dataset from REBASE_DATE onwards
comp2 = pd.DataFrame(index=prices.index)
comp2['Custom Liquid Index'] = liquid_index
comp2['Custom Broad Index']  = broad_index
comp2['Liv-ex 100']          = prices.get('Liv-ex 100')
comp2['Liv-ex 1000']         = prices.get('Liv-ex 1000')
comp2 = comp2[comp2.index >= REBASE_DATE]

# Rebase each column to 100 at REBASE_DATE
for col in comp2.columns:
    comp2[col] = rebase_at(comp2[col], REBASE_DATE)

col_styles = {
    'Custom Liquid Index': (COLOURS['teal'],    2.5, '-'),
    'Custom Broad Index':  (COLOURS['green'],   2.0, '-'),
    'Liv-ex 100':          (COLOURS['purple'],  1.8, '--'),
    'Liv-ex 1000':         (COLOURS['blue'],    1.5, ':'),
}

fig, ax = plt.subplots(figsize=(14, 6))

for col, (colour, lw, ls) in col_styles.items():
    if col not in comp2.columns or comp2[col].isna().all():
        continue
    s = comp2[col].dropna()
    ax.plot(s.index, s.values, color=colour, linewidth=lw, linestyle=ls, label=col, zorder=4)

for sp in STRESS_PERIODS:
    ax.axvspan(pd.Timestamp(sp['start']), pd.Timestamp(sp['end']),
               alpha=0.10, color=sp['colour'], zorder=0)

ax.axhline(100, color=COLOURS['slate'], linewidth=0.7, linestyle=':', alpha=0.5)
ax.set_ylabel(f'Price Index (100 = Jan {REBASE_DATE[:4]})', fontsize=11)
ax.set_xlabel('Date', fontsize=11)

stress_patches = [mpatches.Patch(color=sp['colour'], alpha=0.5, label=sp['label'])
                  for sp in STRESS_PERIODS]
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles + stress_patches, fontsize=9, loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

caption = (
    f'Custom Liquid Index ({len(liquid_lwin7s)} LWIN7s) and Broad Index '
    f'({len(broad_lwin7s)} LWIN7s, vintage >= 1980) vs Liv-ex 100 & 1000. '
    f'All series rebased to 100 at Jan {REBASE_DATE[:4]}. '
    'Divergence from Liv-ex reflects liquidity bias in narrow indices.'
)
fig.text(0.5, -0.02, caption, ha='center', va='top', fontsize=9,
         style='italic', color=COLOURS['slate'])

plt.tight_layout()
out2 = IMAGES_DIR / '02_custom_vs_livex.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
print(f'Saved -> {out2}')
plt.show()

## 10. Chart 3 — Rolling 12-Month Return Spread (Broad Index vs Liv-ex 100)

Rolling 12-month return (%) for the Broad Index minus Liv-ex 100. Positive months
(Broad Index outperforms) are filled teal; negative (Liv-ex 100 outperforms) are filled
purple. The spread reveals when diversifying into less-liquid wines added value.

In [ ]:
if 'Liv-ex 100' not in comp2.columns or comp2['Liv-ex 100'].isna().all():
    print('Liv-ex 100 data unavailable -- skipping spread chart.')
else:
    # Rolling 12m return = price[t] / price[t-12] - 1
    broad_r12  = comp2['Custom Broad Index'].pct_change(12) * 100
    lx100_r12  = comp2['Liv-ex 100'].pct_change(12) * 100
    liquid_r12 = comp2['Custom Liquid Index'].pct_change(12) * 100
    spread     = broad_r12 - lx100_r12

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Panel 1: individual rolling returns
    ax1.plot(broad_r12.index,  broad_r12.values,
             color=COLOURS['green'],  linewidth=1.8, label='Broad Index (12m return %)')
    ax1.plot(liquid_r12.index, liquid_r12.values,
             color=COLOURS['teal'],   linewidth=1.4, linestyle='--',
             label='Liquid Index (12m return %)')
    ax1.plot(lx100_r12.index,  lx100_r12.values,
             color=COLOURS['purple'], linewidth=1.8, label='Liv-ex 100 (12m return %)')
    ax1.axhline(0, color=COLOURS['slate'], linewidth=0.7, linestyle=':')
    ax1.set_ylabel('Rolling 12m return (%)', fontsize=10)
    ax1.legend(fontsize=9, loc='upper left')

    # Panel 2: spread (Broad Index minus Liv-ex 100)
    ax2.fill_between(spread.index, spread.values, 0,
                     where=(spread.values > 0), alpha=0.40, color=COLOURS['teal'],
                     label='Broad Index outperforms')
    ax2.fill_between(spread.index, spread.values, 0,
                     where=(spread.values < 0), alpha=0.40, color=COLOURS['purple'],
                     label='Liv-ex 100 outperforms')
    ax2.plot(spread.index, spread.values,
             color=COLOURS['slate'], linewidth=0.9, alpha=0.6)
    ax2.axhline(0, color=COLOURS['slate'], linewidth=0.8, linestyle=':')
    ax2.set_ylabel('Spread: Broad Index minus Liv-ex 100 (pp)', fontsize=10)
    ax2.set_xlabel('Date', fontsize=11)
    ax2.legend(fontsize=9, loc='upper left')

    for ax in [ax1, ax2]:
        for sp in STRESS_PERIODS:
            ax.axvspan(pd.Timestamp(sp['start']), pd.Timestamp(sp['end']),
                       alpha=0.08, color=sp['colour'], zorder=0)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    caption = (
        'Top panel: rolling 12-month returns for Broad Index, Liquid Index, and Liv-ex 100. '
        'Bottom panel: spread (Broad Index minus Liv-ex 100, pp). '
        'Positive spread (teal) = Broad Index outperforms; negative (purple) = Liv-ex 100 outperforms.'
    )
    fig.text(0.5, -0.02, caption, ha='center', va='top', fontsize=9,
             style='italic', color=COLOURS['slate'])

    print(f'\nSpread summary (Broad Index minus Liv-ex 100, rolling 12m, pp):')
    print(f'  Mean:   {spread.mean():+.2f}')
    print(f'  Std:    {spread.std():.2f}')
    print(f'  Min:    {spread.min():+.2f}')
    print(f'  Max:    {spread.max():+.2f}')

    plt.tight_layout()
    out3 = IMAGES_DIR / '03_spread_vs_livex100.png'
    plt.savefig(out3, dpi=150, bbox_inches='tight')
    print(f'Saved -> {out3}')
    plt.show()

## 11. Chart 4 — Crisis Deep-Dive (GFC and COVID)

Three subplots — one per stress period — each showing the Liquid Index, Broad Index,
Liv-ex 100, and S&P 500 (GBP) rebased to 100 at the window's start date.

The GFC panel is the most instructive: fine wine (both indices) fell less than equities
and recovered faster, but the Broad Index — covering less liquid wines — behaved
differently from the Liquid Index, illustrating intra-wine heterogeneity.

In [ ]:
# Build aligned series for crisis analysis
crisis_series = {
    'Custom Liquid Index': liquid_index.rename('Custom Liquid Index'),
    'Custom Broad Index':  broad_index.rename('Custom Broad Index'),
    'Liv-ex 100':    prices.get('Liv-ex 100', pd.Series(dtype=float, name='Liv-ex 100')),
    'S&P 500 (GBP)': prices['S&P 500 (GBP)'],
}

series_colours = {
    'Custom Liquid Index': COLOURS['teal'],
    'Custom Broad Index':  COLOURS['green'],
    'Liv-ex 100':    COLOURS['purple'],
    'S&P 500 (GBP)': COLOURS['blue'],
}

PLOT_BUFFERS = {
    'GFC 2008':        ('2006-01-01', '2012-12-31'),
    'COVID 2020':      ('2019-01-01', '2022-06-30'),
    '2022 Rate Rises': ('2020-07-01', '2024-06-30'),
}

n_panels = len(STRESS_PERIODS)
fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6), sharey=False)

for ax, sp in zip(axes, STRESS_PERIODS):
    pstart, pend = PLOT_BUFFERS.get(sp['label'], (sp['start'], sp['end']))
    ps = pd.Timestamp(pstart)
    pe = pd.Timestamp(pend)
    cs = pd.Timestamp(sp['start'])
    ce = pd.Timestamp(sp['end'])

    for name, s in crisis_series.items():
        if s is None or (hasattr(s, 'isna') and s.isna().all()):
            continue
        window = s[(s.index >= ps) & (s.index <= pe)].dropna()
        if len(window) < 3:
            continue
        colour = series_colours.get(name, COLOURS['slate'])
        lw = 2.2 if 'Custom' in name else 1.6
        indexed = window / window.iloc[0] * 100
        ax.plot(indexed.index, indexed.values, color=colour, linewidth=lw, label=name)

    ax.axvspan(cs, ce, alpha=0.14, color=sp['colour'], zorder=0, label=sp['label'])
    ax.axhline(100, color=COLOURS['slate'], linewidth=0.7, linestyle=':', alpha=0.5)
    ax.set_title(sp['label'], fontsize=11, fontweight='bold')
    ax.set_ylabel('Index (window start = 100)', fontsize=9)
    ax.legend(fontsize=8, loc='lower left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', rotation=30)

caption = (
    'Custom Liquid Index, Broad Index (vintage >= 1980), Liv-ex 100, and S&P 500 (GBP) '
    'rebased to 100 at each crisis window start. Shaded region = defined stress period. '
    'Fine wine indices typically show smaller drawdowns and faster recoveries than equities.'
)
fig.text(0.5, -0.03, caption, ha='center', va='top', fontsize=9,
         style='italic', color=COLOURS['slate'])

plt.tight_layout()
out4 = IMAGES_DIR / '04_crisis_deepdive.png'
plt.savefig(out4, dpi=150, bbox_inches='tight')
print(f'Saved -> {out4}')
plt.show()

## 12. Chart 5 — Crisis Bar Comparison

Grouped bar chart: 3 stress periods x 5 assets (Liquid Index, Broad Index, Liv-ex 100,
Liv-ex 1000, S&P 500 (GBP)). Each bar shows the total return during the stress period.

This chart is the key policy-relevant output: it directly answers "how did fine wine hold
up during each crisis, relative to equities and Liv-ex benchmarks?"

In [ ]:
bar_series = {
    'Custom Liquid Index': liquid_index,
    'Custom Broad Index':  broad_index,
    'Liv-ex 100':          prices.get('Liv-ex 100'),
    'Liv-ex 1000':         prices.get('Liv-ex 1000'),
    'S&P 500 (GBP)':       prices['S&P 500 (GBP)'],
}

bar_colours = {
    'Custom Liquid Index': COLOURS['teal'],
    'Custom Broad Index':  COLOURS['green'],
    'Liv-ex 100':          COLOURS['purple'],
    'Liv-ex 1000':         COLOURS['magenta'],
    'S&P 500 (GBP)':       COLOURS['blue'],
}

# Compute total return for each asset x stress period
crisis_stats = []
for sp in STRESS_PERIODS:
    row = {'period': sp['label']}
    for name, s in bar_series.items():
        if s is None or (hasattr(s, 'isna') and s.isna().all()):
            row[name] = np.nan
            continue
        window = s[(s.index >= sp['start']) & (s.index <= sp['end'])].dropna()
        if len(window) < 2:
            row[name] = np.nan
            continue
        ret = (float(window.iloc[-1]) - float(window.iloc[0])) / float(window.iloc[0]) * 100
        row[name] = round(ret, 1)
    crisis_stats.append(row)

crisis_df = pd.DataFrame(crisis_stats).set_index('period')
print('=== Total return during each stress period ===')
display(crisis_df)

In [ ]:
asset_names = [k for k in bar_series if k in crisis_df.columns]
n_periods   = len(crisis_df)
n_assets    = len(asset_names)
x           = np.arange(n_periods)
total_width = 0.80
bar_width   = total_width / n_assets

fig, ax = plt.subplots(figsize=(13, 6))

for i, name in enumerate(asset_names):
    vals   = crisis_df[name].values.astype(float)
    offset = (i - n_assets / 2 + 0.5) * bar_width
    colour = bar_colours.get(name, COLOURS['slate'])
    bars   = ax.bar(x + offset, vals, width=bar_width,
                    color=colour, label=name, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        if np.isnan(val):
            continue
        va   = 'bottom' if val >= 0 else 'top'
        ypos = val + (0.6 if val >= 0 else -0.6)
        ax.text(
            bar.get_x() + bar.get_width() / 2, ypos,
            f'{val:+.1f}%',
            ha='center', va=va, fontsize=7.5, color=COLOURS['slate']
        )

ax.axhline(0, color=COLOURS['slate'], linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(crisis_df.index, fontsize=10)
ax.set_ylabel('Total return during stress period (%)', fontsize=10)
ax.legend(fontsize=9, loc='best')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:+.0f}%'))

caption = (
    'Total return (%) during each stress period for Custom Liquid Index, Broad Index '
    '(vintage >= 1980, chain-link), Liv-ex 100, Liv-ex 1000, and S&P 500 (GBP). '
    'Fine wine indices typically show shallower drawdowns than equities during crises.'
)
fig.text(0.5, -0.02, caption, ha='center', va='top', fontsize=9,
         style='italic', color=COLOURS['slate'])

plt.tight_layout()
out5 = IMAGES_DIR / '05_crisis_bar_comparison.png'
plt.savefig(out5, dpi=150, bbox_inches='tight')
print(f'Saved -> {out5}')
plt.show()

## 13. Data Quality Assertions

All checks must pass before this notebook is considered complete.

In [ ]:
errors = []

# 1. All 5 PNGs exist with minimum file size
REQUIRED_IMAGES = [
    '01_constituent_series.png',
    '02_custom_vs_livex.png',
    '03_spread_vs_livex100.png',
    '04_crisis_deepdive.png',
    '05_crisis_bar_comparison.png',
]
for fname in REQUIRED_IMAGES:
    p = IMAGES_DIR / fname
    if not p.exists():
        errors.append(f'Missing image: {fname}')
    elif p.stat().st_size < 1000:
        errors.append(f'Image too small (likely empty): {fname}')

# 2. Vintage filter applied -- all vintages must be >= 1980
if latent_raw['vintage'].min() < 1980:
    errors.append(f'Vintage filter not applied -- min vintage: {latent_raw["vintage"].min()}')

# 3. Both indices have enough valid months
for name, idx in [('Liquid', liquid_index), ('Broad', broad_index)]:
    cnt = int(idx.notna().sum())
    if cnt < 60:
        errors.append(f'{name} Index has only {cnt} valid months (need >= 60)')

# 4. Indices start near 100 at rebase month
for name, idx in [('Liquid', liquid_index), ('Broad', broad_index)]:
    first = float(idx.dropna().iloc[0])
    if not (80 <= first <= 120):
        errors.append(f'{name} Index first value {first:.1f} not near 100 -- rebase failed')

# 5. At least 10 LWIN7s in liquid index
if len(liquid_lwin7s) < 10:
    errors.append(f'Only {len(liquid_lwin7s)} liquid LWIN7s (need >= 10)')

# 6. Broad index must include more wines than liquid index
if len(broad_lwin7s) <= len(liquid_lwin7s):
    errors.append(
        f'Broad Index ({len(broad_lwin7s)} LWIN7s) must be larger than '
        f'Liquid Index ({len(liquid_lwin7s)} LWIN7s)'
    )

# 7. Liv-ex 100 column detected
if LX100_COL is None:
    errors.append('Liv-ex 100 column not detected in livex parquet')

# 8. Crisis bar chart has data for at least 2 assets across all periods
non_null_cols = crisis_df.notna().any(axis=0)
if non_null_cols.sum() < 2:
    errors.append(f'Crisis bar chart has data for only {non_null_cols.sum()} assets (need >= 2)')

# 9. MotherDuck data loaded (not parquet)
if 'latent_raw' not in dir() or len(latent_raw) == 0:
    errors.append('latent_raw not populated -- MotherDuck query may have failed')

# 10. Images directory exists
if not IMAGES_DIR.is_dir():
    errors.append(f'Images directory missing: {IMAGES_DIR}')

if errors:
    for err in errors:
        print(f'FAIL: {err}')
    raise AssertionError('Data quality checks failed -- see output above')
else:
    print('All data quality assertions PASSED.')
    print(f'  Data source:               dev_winefi_raf.ml.ml_latent_prices_historic')
    print(f'  Vintage filter:            >= 1980 (min: {latent_raw["vintage"].min()})')
    print(f'  LWIN7 count (all):         {len(all_lwin7s)}')
    print(f'  Liquid Index LWIN7s:       {len(liquid_lwin7s)} (top-{LIQUID_TOP_N})')
    print(f'  Broad Index LWIN7s:        {len(broad_lwin7s)} (>= {MIN_MONTHS_BROAD}m coverage)')
    print(f'  Liquid rebase month:       {LIQUID_REBASE_MONTH.date()}')
    print(f'  Broad rebase month:        {BROAD_REBASE_MONTH.date()}')
    print(f'  Liv-ex 100 column:         {LX100_COL}')
    print(f'  Liv-ex 1000 column:        {LX1000_COL}')
    print(f'  Images saved:')
    for fname in REQUIRED_IMAGES:
        size_kb = (IMAGES_DIR / fname).stat().st_size / 1024
        print(f'    {fname}  ({size_kb:.1f} KB)')

## Summary

| Chart | File | Description |
|-------|------|-------------|
| 1 | `01_constituent_series.png` | Top-10 chain-link normalised LWIN7 series + Liquid Index composite |
| 2 | `02_custom_vs_livex.png` | Liquid & Broad indices vs Liv-ex 100 & 1000, rebased Jan 2005 |
| 3 | `03_spread_vs_livex100.png` | Rolling 12m return spread: Broad Index minus Liv-ex 100 |
| 4 | `04_crisis_deepdive.png` | 3 crisis windows: Liquid/Broad vs Liv-ex 100 vs S&P 500 (GBP) |
| 5 | `05_crisis_bar_comparison.png` | Grouped bar: 3 stress periods x 5 assets |

### Index methodology
- **Data source**: `dev_winefi_raf.ml.ml_latent_prices_historic` (MotherDuck)
- **Vintage filter**: >= 1980 applied in SQL
- **Model**: Latent trade price (MCMC/Kalman state-space)
- **Aggregation**: LWIN11 -> LWIN7 by median `price_latent` across vintages per month
- **Chain-link normalisation**: each LWIN7 = 100 at first observation; only post-entry appreciation contributes
- **Liquid Index**: top-50 LWIN7s by months of coverage (equal-weighted median)
- **Broad Index**: all LWIN7s with >= 24 months coverage (equal-weighted median)
- **Rebase**: earliest month with >= 10 (Liquid) / >= 5 (Broad) contributing LWIN7s
- **Stress periods**: GFC 2008, COVID 2020, 2022 Rate Rises

### Key academic references
- Dimson, Rousseau & Spaenjers (2015): ~4.1% real annual return 1900-2012
- Case & Shiller (1987/1989): repeat-sales / chain-link for illiquid assets
- Goetzmann (1992): illiquidity-induced downward bias in correlation estimates
- Masset & Henderson (2010): fine wine diversification benefits

---
*Data source*: `dev_winefi_raf.ml.ml_latent_prices_historic` (MotherDuck)  
*Also depends on*: `01_data_setup.ipynb` (Liv-ex and comparison parquet files in `../data/`)  
*Outputs*: `../images/custom_indices/` -- 5 PNG charts